# Mars image anomaly detection

> **ARCHIVED / NOT THE FINAL SUBMISSION**
>
> This notebook is retained only for experiment and development provenance.
> It contains an earlier pre-final pipeline and must not be used as the
> competition deliverable or as the source of canonical results.
>
> The sole canonical evaluated submission is:
> `Mars_HiRISE_Submission.ipynb`.
>
> The canonical detector is the v3 pipeline documented in the submission
> notebook and final report.

## NSSC 2026 — my experiments and final analysis

My aim is to find unusual crops using a convolutional autoencoder trained from scratch, followed by Isolation Forest on its one-dimensional latent vector.

I started with a reconstruction baseline, changed the loss, and tested a larger vector. The first candidate list was dominated by very bright crops and changed across repeated runs. I added two preprocessing experiments to investigate that instead of assuming that the highest scores were genuine anomalies.

**Execution note:** I prepared this notebook for Colab or Kaggle. The saved experiment results and the recorded notebook execution were produced locally on an RTX 3050. The setup below prints the actual runtime; I have not relabelled the local results as a Colab execution.

There are no hidden anomaly labels available to me. I can measure reconstruction, nuisance sensitivity and repeatability, but not detection precision or recall.


## 0. Setup
On Colab I would select **Runtime → Change runtime type → GPU**. I keep the notebook and supplied data private. I use no outside training images or pretrained feature extractors.


In [ ]:
from pathlib import Path
import os, sys, json, importlib.util, subprocess
if Path('/content').exists():
    ROOT = Path('/content/mars_project')
elif Path('/kaggle/working').exists():
    ROOT = Path('/kaggle/working/mars_project')
else:
    ROOT = Path.cwd()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
packages = {'numpy':'numpy', 'pandas':'pandas', 'PIL':'Pillow', 'scipy':'scipy',
            'sklearn':'scikit-learn', 'matplotlib':'matplotlib',
            'statsmodels':'statsmodels', 'torch':'torch'}
missing = [p for m,p in packages.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage, Markdown
print('Folder:', ROOT)
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU')


## 0.1 Data and preprocessing
I keep all crops from a source observation in the same split. Otherwise, similar acquisition conditions could appear in both training and validation. The main split has 120 training, 25 validation and 27 calibration sources.

I compare four input options:
- **border_fill (v6):** mask border-connected values at most 4/255, expand by two pixels and extend nearest valid standardized values. This can create artificial stripes.
- **raw:** original grayscale divided by 255.
- **contrast:** subtract the crop mean, divide by its standard deviation, then clip 0.5 + 0.15 × z to [0,1].
- **footprint:** estimate those statistics outside exact-black regions connected to the image border, then fill those regions with 0.5.

I floor the standard deviation at 1/255 for nearly constant crops. The footprint mask is my heuristic, not supplied ground truth. It could affect a border-connected shadow; normalization can also remove meaningful albedo differences.


In [ ]:
"""Explicit, per-image photometric ablations; no fitted dataset statistics."""
import numpy as np
from scipy.ndimage import label, binary_dilation, distance_transform_edt
MODES = ('raw', 'contrast', 'footprint', 'border_fill')

def border_zero_mask(array):
    regions, _ = label(array == 0)
    border = np.unique(np.concatenate([regions[0], regions[-1], regions[:, 0], regions[:, -1]]))
    return np.isin(regions, border[border != 0])

def border_near_black_mask(array):
    """Fixed JPEG-tolerant border heuristic; not a ground-truth validity mask."""
    regions, _ = label(array <= 4 / 255)
    edge = np.unique(np.concatenate([regions[0], regions[-1], regions[:, 0], regions[:, -1]]))
    mask = np.isin(regions, edge[edge != 0])
    return binary_dilation(mask, iterations=2) if mask.any() else mask

def preprocess_array(array, mode='raw'):
    if mode not in MODES:
        raise ValueError(f'Unknown preprocessing: {mode}')
    x = np.asarray(array, dtype=np.float32)
    if x.ndim != 2 or not np.isfinite(x).all():
        raise ValueError('Expected finite 2D image')
    if mode == 'raw':
        return x
    excluded = border_near_black_mask(x) if mode == 'border_fill' else border_zero_mask(x) if mode == 'footprint' else np.zeros(x.shape, dtype=bool)
    values = x[~excluded]
    if not len(values):
        return np.full_like(x, 0.5)
    mean = float(values.mean())
    std = max(float(values.std()), 1 / 255)
    result = np.clip(0.5 + 0.15 * (x - mean) / std, 0, 1).astype(np.float32)
    if mode == 'border_fill' and excluded.any():
        nearest = distance_transform_edt(excluded, return_distances=False, return_indices=True)
        result[excluded] = result[tuple(nearest[:, excluded])]
    else:
        result[excluded] = 0.5
    return result


In [ ]:
from pathlib import Path
import io
from zipfile import ZipFile
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset

def prepare_archive(archive, destination):
    """Extract only the known dataset structure, never arbitrary archive paths."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with ZipFile(archive) as outer:
        for name in ('crop_metadata_index.csv', 'source_image_metadata.csv'):
            (destination / name).write_bytes(outer.read('DATASETS/' + name))
        payload = outer.read('DATASETS/images.zip')
    expected = set(pd.read_csv(destination / 'crop_metadata_index.csv').filename)
    seen = set()
    image_dir = destination / 'images'
    image_dir.mkdir(exist_ok=True)
    with ZipFile(io.BytesIO(payload)) as inner:
        for entry in inner.infolist():
            if entry.is_dir():
                continue
            name = Path(entry.filename).name
            if name not in expected or name in seen:
                raise ValueError(f'Unexpected or duplicate image: {name}')
            seen.add(name)
            target = image_dir / name
            if not target.exists():
                target.write_bytes(inner.read(entry))
    if seen != expected:
        raise ValueError('Archive and crop index do not match')
    return destination

def load_manifest(data_dir, seed=2026):
    data_dir = Path(data_dir)
    crops = pd.read_csv(data_dir / 'crop_metadata_index.csv')
    sources = pd.read_csv(data_dir / 'source_image_metadata.csv')
    if not crops.filename.is_unique or not sources.source_image_id.is_unique:
        raise ValueError('Duplicate metadata join keys')
    frame = crops.merge(sources, on='source_image_id', how='left', validate='many_to_one')
    if frame.isna().any().any():
        raise ValueError('Missing metadata')
    groups = np.array(sorted(frame.source_image_id.unique()))
    if len(groups) < 10:
        raise ValueError('At least 10 source groups required')
    np.random.default_rng(seed).shuffle(groups)
    n_train, n_val = (int(0.7 * len(groups)), int(0.15 * len(groups)))
    assignment = {g: 'train' if i < n_train else 'validation' if i < n_train + n_val else 'calibration' for i, g in enumerate(groups)}
    frame['split'] = frame.source_image_id.map(assignment)
    frame['path'] = [str(data_dir / 'images' / name) for name in frame.filename]
    if not all((Path(p).is_file() for p in frame.path)):
        raise FileNotFoundError('Some indexed image files are missing')
    return frame

class CropDataset(Dataset):

    def __init__(self, frame, preprocessing='raw'):
        self.frame = frame.reset_index(drop=True)
        if preprocessing not in MODES:
            raise ValueError('Unknown preprocessing mode')
        self.preprocessing = preprocessing

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.path) as image:
            if image.mode != 'L' or image.size != (227, 227):
                raise ValueError(f'Unexpected image contract: {row.filename}')
            array = np.array(image, dtype=np.float32) / 255.0
        return (torch.from_numpy(preprocess_array(array, self.preprocessing)).unsqueeze(0), index)


In [ ]:
DATA = ROOT / 'data'
ARCHIVE = None  # Set an explicit outer dataset ZIP if necessary.
if not (DATA / 'crop_metadata_index.csv').exists():
    candidates = list(ROOT.glob('DATASETS-*.zip'))
    if Path('/kaggle/input').exists():
        candidates += list(Path('/kaggle/input').rglob('DATASETS-*.zip'))
    if ARCHIVE is None and candidates:
        ARCHIVE = candidates[0]
    if ARCHIVE is None and Path('/content').exists():
        from google.colab import files
        uploaded = files.upload()
        archives = [Path(n) for n in uploaded if n.endswith('.zip')]
        if len(archives) == 1:
            ARCHIVE = archives[0]
    if ARCHIVE is None:
        raise FileNotFoundError('Set ARCHIVE to the supplied DATASETS outer ZIP.')
    prepare_archive(ARCHIVE, DATA)
manifest = load_manifest(DATA, seed=2026)
assert len(manifest) == 10422 and manifest.filename.is_unique
assert manifest.groupby('source_image_id').split.nunique().max() == 1
display(manifest.groupby('split').agg(crops=('filename','size'),
                                     sources=('source_image_id','nunique')))
display(manifest[['latitude','longitude','sun_angle','season','resolution']].nunique())


My complete audit decoded 10,422 crops as 227 × 227 grayscale images, checked all joins and found no exact decoded-pixel duplicates. This does not rule out near duplicates or overlapping terrain. Longitude has only two values, and resolution units are unspecified.

Before training, I check preprocessing on fixed validation images. These are sanity examples, not anomaly labels.


In [ ]:
preview = manifest[manifest.split == 'validation'].iloc[[0,1,6,7]]
fig, axes = plt.subplots(4,3,figsize=(9,11))
for i,(_,row) in enumerate(preview.iterrows()):
    raw,_ = CropDataset(pd.DataFrame([row]))[0]
    for j,mode in enumerate(['raw','contrast','footprint']):
        axes[i,j].imshow(preprocess_array(raw[0].numpy(),mode),cmap='gray',vmin=0,vmax=1)
        axes[i,j].set_title(f'{row.filename}: {mode}',fontsize=9)
        axes[i,j].axis('off')
fig.tight_layout()
display(fig)
plt.close(fig)


## Phase 1.1 — My autoencoder
I use five stride-two convolution blocks with channels 16, 32, 64, 96 and 128. Spatial sizes become 114, 57, 29, 15 and 8. A linear layer maps the final feature map to 128 or 256 numbers. The decoder must reconstruct through this vector; there is no skip connection around it.

GroupNorm supports my small batches. Resize-convolution gives explicit output sizes and avoids uneven transposed-convolution overlap. All weights start randomly.

## Phase 1.2 — Loss
v1 uses MSE. For v2-v6 I use 0.9 MSE + 0.1 (1 − SSIM), because the baseline smoothed fine texture. SSIM uses an 11 × 11 Gaussian window and direct pixel statistics, without a pretrained network. I report common metrics separately.


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

class ConvAutoencoder(nn.Module):
    """No pretrained weights, external features, or bottleneck bypass."""

    def __init__(self, latent_dim=128):
        super().__init__()
        channels = (1, 16, 32, 64, 96, 128)
        blocks = []
        for cin, cout in zip(channels[:-1], channels[1:]):
            blocks.extend([nn.Conv2d(cin, cout, 3, stride=2, padding=1), nn.GroupNorm(8, cout), nn.SiLU()])
        self.encoder = nn.Sequential(*blocks)
        self.to_latent = nn.Linear(128 * 8 * 8, latent_dim)
        self.from_latent = nn.Linear(latent_dim, 128 * 8 * 8)
        self.decoder = nn.ModuleList()
        for cin, cout in zip((128, 96, 64, 32, 16), (96, 64, 32, 16, 1)):
            layers = [nn.Conv2d(cin, cout, 3, padding=1)]
            layers += [nn.Sigmoid()] if cout == 1 else [nn.GroupNorm(8, cout), nn.SiLU()]
            self.decoder.append(nn.Sequential(*layers))

    def encode(self, x):
        return self.to_latent(self.encoder(x).flatten(1))

    def decode(self, z):
        x = self.from_latent(z).reshape(-1, 128, 8, 8)
        for size, block in zip((15, 29, 57, 114, 227), self.decoder):
            x = block(F.interpolate(x, size=(size, size), mode='bilinear', align_corners=False))
        return x

    def forward(self, x):
        z = self.encode(x)
        return (self.decode(z), z)

def ssim_per_image(x, y):
    """11x11 Gaussian-window SSIM, sigma 1.5, data range 1, valid windows."""
    x, y = (x.float(), y.float())
    coords = torch.arange(11, device=x.device, dtype=x.dtype) - 5
    weights = torch.exp(-coords.square() / (2 * 1.5 ** 2))
    weights = weights / weights.sum()

    def smooth(a):
        a = F.conv2d(a, weights.reshape(1, 1, 1, 11))
        return F.conv2d(a, weights.reshape(1, 1, 11, 1))
    mx, my = (smooth(x), smooth(y))
    vx = (smooth(x * x) - mx * mx).clamp_min(0)
    vy = (smooth(y * y) - my * my).clamp_min(0)
    cov = smooth(x * y) - mx * my
    result = (2 * mx * my + 0.01 ** 2) * (2 * cov + 0.03 ** 2) / ((mx * mx + my * my + 0.01 ** 2) * (vx + vy + 0.03 ** 2))
    return result.mean((1, 2, 3))

def loss_components(x, reconstruction, structural_weight=0.0, gradient_weight=0.0):
    x, reconstruction = (x.float(), reconstruction.float())
    mse = (x - reconstruction).square().mean((1, 2, 3))
    with torch.set_grad_enabled(torch.is_grad_enabled() and structural_weight > 0):
        ssim = ssim_per_image(x, reconstruction)
    dx = x[:, :, :, 1:] - x[:, :, :, :-1] - (reconstruction[:, :, :, 1:] - reconstruction[:, :, :, :-1])
    dy = x[:, :, 1:, :] - x[:, :, :-1, :] - (reconstruction[:, :, 1:, :] - reconstruction[:, :, :-1, :])
    gradient = 0.5 * (dx.abs().mean((1, 2, 3)) + dy.abs().mean((1, 2, 3)))
    loss = (1 - structural_weight) * mse
    if structural_weight:
        loss = loss + structural_weight * (1 - ssim)
    if gradient_weight:
        loss = loss + gradient_weight * gradient
    return {'loss': loss.mean(), 'mse': mse.mean(), 'ssim': ssim.mean(), 'gradient': gradient.mean()}


In [ ]:
torch.set_num_threads(4)
for dimension in [128,256]:
    example_model = ConvAutoencoder(dimension)
    with torch.no_grad():
        reconstruction,vector = example_model(torch.zeros(1,1,227,227))
    print(dimension,'dimensions;',sum(p.numel() for p in example_model.parameters()),
          'parameters;',tuple(vector.shape),'latent;',tuple(reconstruction.shape),'output')
del example_model


## Training and saved evidence
Each epoch saves metrics and a fixed reconstruction panel. I select the checkpoint using that run's validation objective. A resumed run must use the same configuration. The plotting and training functions are visible here so this notebook runs without a separate Python package.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def reconstruction_panel(original, reconstruction, path, input_label='Original'):
    original = original.detach().float().cpu().numpy()[:, 0]
    reconstruction = reconstruction.detach().float().cpu().numpy()[:, 0]
    n = len(original)
    fig, axes = plt.subplots(3, n, figsize=(2 * n, 6), squeeze=False)
    error = np.abs(original - reconstruction)
    vmax = max(float(error.max()), 1e-06)
    for j in range(n):
        axes[0, j].imshow(original[j], cmap='gray', vmin=0, vmax=1)
        axes[1, j].imshow(reconstruction[j], cmap='gray', vmin=0, vmax=1)
        im = axes[2, j].imshow(error[j], cmap='inferno', vmin=0, vmax=vmax)
        for i in range(3):
            axes[i, j].set_xticks([])
            axes[i, j].set_yticks([])
    for i, label in enumerate([input_label, 'Reconstruction', 'Absolute error']):
        axes[i, 0].set_ylabel(label)
    fig.colorbar(im, ax=axes[2, :].tolist(), label='Absolute intensity error [0,1]', fraction=0.02)
    fig.savefig(path, dpi=130, bbox_inches='tight')
    plt.close(fig)

def score_plots(frame, calibration, path):
    scores = frame.novelty_score.to_numpy()
    threshold = calibration['threshold']
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for split, g in frame.groupby('split'):
        axes[0].hist(g.novelty_score, bins=50, density=True, histtype='step', label=split)
    axes[0].axvline(threshold, color='red', label='Calibrated fence')
    axes[0].legend(fontsize=8)
    axes[0].set(xlabel='Novelty score (higher = more anomalous)', ylabel='Density', title='Score distributions by split')
    axes[1].plot(np.arange(len(scores)), np.sort(scores))
    axes[1].axhline(threshold, color='red')
    axes[1].axhspan(*calibration['bootstrap_interval_95'], color='red', alpha=0.1)
    axes[1].set(xlabel='Sorted crop index', ylabel='Novelty score', title='Sorted scores and threshold uncertainty')
    axes[2].plot(np.sort(scores), np.arange(1, len(scores) + 1) / len(scores))
    axes[2].axvline(threshold, color='red')
    axes[2].set(xlabel='Novelty score', ylabel='Empirical cumulative fraction', title='Empirical score distribution')
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

def heatmap_panel(original, reconstruction, row, threshold, path):
    original, reconstruction = (np.asarray(original), np.asarray(reconstruction))
    error = np.abs(original - reconstruction)
    fig, axes = plt.subplots(1, 4, figsize=(13, 3.8))
    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])
    axes[0].imshow(original, cmap='gray', vmin=0, vmax=1)
    axes[0].set_title('Original')
    axes[1].imshow(reconstruction, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title('Reconstruction')
    im = axes[2].imshow(error, cmap='inferno', vmin=0, vmax=1)
    axes[2].set_title('Absolute error')
    axes[3].imshow(original, cmap='gray', vmin=0, vmax=1)
    axes[3].imshow(error, cmap='inferno', vmin=0, vmax=1, alpha=0.55)
    axes[3].set_title('Error overlay')
    fig.colorbar(im, ax=axes.tolist(), fraction=0.018, pad=0.02, label='Error [0,1]; same scale for all candidates')
    fig.suptitle(f'{row.filename} | {row.source_image_id} | novelty {row.novelty_score:.4f} > {threshold:.4f}\nMaximum absolute error {error.max():.3f}; mean {error.mean():.3f}', fontsize=10)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)

def standardized_heatmap_panel(raw, standardized, reconstruction, row, threshold, path, mode):
    error = np.abs(standardized - reconstruction)
    fig, axes = plt.subplots(1, 5, figsize=(15, 4))
    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])
    for ax, value, title in zip(axes[:3], [raw, standardized, reconstruction], ['Raw crop', f'Model input: {mode}', 'Reconstruction']):
        ax.imshow(value, cmap='gray', vmin=0, vmax=1)
        ax.set_title(title, fontsize=10)
    im = axes[3].imshow(error, cmap='inferno', vmin=0, vmax=1)
    axes[3].set_title('Model-input error', fontsize=10)
    axes[4].imshow(raw, cmap='gray', vmin=0, vmax=1)
    axes[4].imshow(error, cmap='inferno', vmin=0, vmax=1, alpha=0.55)
    axes[4].set_title('Error on raw scene', fontsize=10)
    fig.colorbar(im, ax=axes.tolist(), fraction=0.016, pad=0.02, label='Standardized-domain error [0,1]')
    fig.suptitle(f'{row.filename} | {row.source_image_id} | novelty {row.novelty_score:.4f} > {threshold:.4f}\nError compares standardized input with reconstruction; not direct forest attribution', fontsize=10)
    fig.savefig(path, dpi=160, bbox_inches='tight')
    plt.close(fig)


In [ ]:
import argparse
from dataclasses import asdict, dataclass
from pathlib import Path
import json
import random
import time
import platform
from datetime import datetime
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

@dataclass
class TrainConfig:
    version: str = 'v1'
    latent_dim: int = 128
    epochs: int = 15
    batch_size: int = 16
    learning_rate: float = 0.0003
    structural_weight: float = 0.0
    gradient_weight: float = 0.0
    seed: int = 2026
    split_seed: int = 2026
    preprocessing: str = 'raw'
    workers: int = 0
    patience: int = 5
    max_batches: int = 0
    augment: bool = False

def normalized_config(saved):
    result = asdict(TrainConfig())
    result.update(saved)
    result['split_seed'] = saved.get('split_seed', saved.get('seed', 2026))
    result.setdefault('augment', False)
    result.setdefault('gradient_weight', 0.0)
    return result

def validate_config(saved, requested, resume=False):
    prior = normalized_config(saved)
    current = asdict(requested)
    keys = set(current) - {'version', 'workers'}
    if resume:
        keys.remove('epochs')
    for key in sorted(keys):
        if prior[key] != current[key]:
            raise ValueError(f'Configuration mismatch: {key}: saved={prior[key]!r}, requested={current[key]!r}')

def ensure_training(data_dir, output_dir, config):
    """Validate cached runs before reuse, or train/resume the requested configuration."""
    output_dir = Path(output_dir)
    if (output_dir / 'config.json').exists():
        validate_config(json.loads((output_dir / 'config.json').read_text()), config)
    status = output_dir / 'training_status.json'
    if status.exists():
        expected = 'smoke_complete' if config.max_batches else 'complete'
        if json.loads(status.read_text())['status'] != expected:
            raise ValueError('Unexpected cached training status')
        checkpoint = torch.load(output_dir / 'best.pt', map_location='cpu', weights_only=False)
        validate_config(checkpoint['config'], config)
        return
    run_training(data_dir, output_dir, config, resume=(output_dir / 'last.pt').exists())

def atomic_checkpoint(state, destination):
    """Keep the previous checkpoint intact if serialization fails partway."""
    destination = Path(destination)
    temporary = destination.with_suffix('.tmp.pt')
    torch.save(state, temporary)
    temporary.replace(destination)

def run_training(data_dir, output_dir, config, resume=False):
    if config.epochs < 1 or not 0 <= config.structural_weight <= 1 or config.gradient_weight < 0:
        raise ValueError('Invalid training configuration')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    last = output_dir / 'last.pt'
    if last.exists() and (not resume):
        raise FileExistsError(f'{last} already exists; choose a new run directory or resume')
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(config.seed)
    torch.set_num_threads(4)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    manifest = load_manifest(data_dir, config.split_seed)
    loaders = {split: DataLoader(CropDataset(manifest[manifest.split == split], config.preprocessing), batch_size=config.batch_size, shuffle=split == 'train', num_workers=config.workers, pin_memory=device.type == 'cuda') for split in ['train', 'validation']}
    model = ConvAutoencoder(config.latent_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.0001)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
    start, best, stale, history = (0, float('inf'), 0, [])
    if resume:
        checkpoint = torch.load(last, map_location=device, weights_only=False)
        validate_config(checkpoint['config'], config, resume=True)
        model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        scaler.load_state_dict(checkpoint['scaler'])
        start, best, stale = (checkpoint['epoch'] + 1, checkpoint['best'], checkpoint['stale'])
        history = checkpoint['history']
        torch.set_rng_state(checkpoint['torch_rng'].cpu())
        if device.type == 'cuda' and checkpoint['cuda_rng'] is not None:
            torch.cuda.set_rng_state_all([s.cpu() for s in checkpoint['cuda_rng']])
        np.random.set_state(checkpoint['numpy_rng'])
        random.setstate(checkpoint['python_rng'])
    manifest.drop(columns='path').to_csv(output_dir / 'split_manifest.csv', index=False)
    config_path = output_dir / 'config.json'
    config_path.write_text(json.dumps(asdict(config), indent=2))
    environment = {'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__, 'pandas': pd.__version__, 'device': str(device), 'gpu': torch.cuda.get_device_name() if device.type == 'cuda' else None, 'parameters': sum((p.numel() for p in model.parameters())), 'smoke_run': config.max_batches > 0}
    (output_dir / 'environment.json').write_text(json.dumps(environment, indent=2))
    print(json.dumps(environment), flush=True)

    def record_event(event, **details):
        with (output_dir / 'events.jsonl').open('a', encoding='utf-8') as stream:
            stream.write(json.dumps({'time': datetime.now().astimezone().isoformat(), 'event': event, **details}) + '\n')
    record_event('resumed' if resume else 'started', completed_epochs=start, config=asdict(config), device=str(device))
    saved_rng = torch.get_rng_state() if resume else None
    fixed = next(iter(loaders['validation']))[0][:8].to(device)
    if saved_rng is not None:
        torch.set_rng_state(saved_rng)
    for epoch in range(start, config.epochs):
        then = time.perf_counter()
        row = {'epoch': epoch + 1}
        for split, loader in loaders.items():
            training = split == 'train'
            model.train(training)
            sums = dict.fromkeys(['loss', 'mse', 'ssim', 'gradient'], 0.0)
            count = 0
            for batch_index, (x, _) in enumerate(loader):
                if config.max_batches and batch_index >= config.max_batches:
                    break
                x = x.to(device, non_blocking=True)
                if training and config.augment:
                    if torch.rand(1) > 0.5:
                        x = torch.flip(x, [2])
                    if torch.rand(1) > 0.5:
                        x = torch.flip(x, [3])
                    rot = torch.randint(0, 4, (1,)).item()
                    if rot > 0:
                        x = torch.rot90(x, k=rot, dims=[2, 3])
                with torch.set_grad_enabled(training):
                    with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
                        reconstruction, _ = model(x)
                    with torch.autocast(device_type=device.type, enabled=False):
                        metrics = loss_components(x, reconstruction, config.structural_weight, config.gradient_weight)
                    if not torch.isfinite(metrics['loss']):
                        raise RuntimeError('Nonfinite training loss')
                    if training:
                        optimizer.zero_grad(set_to_none=True)
                        scaler.scale(metrics['loss']).backward()
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                        scaler.step(optimizer)
                        scaler.update()
                for key, value in metrics.items():
                    sums[key] += float(value.detach()) * len(x)
                count += len(x)
            row.update({split + '_' + key: value / count for key, value in sums.items()})
            row[split + '_images'] = count
        row['seconds'] = time.perf_counter() - then
        history.append(row)
        improved = row['validation_loss'] < best
        best = min(best, row['validation_loss'])
        stale = 0 if improved else stale + 1
        state = {'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(), 'config': asdict(config), 'epoch': epoch, 'best': best, 'stale': stale, 'history': history, 'torch_rng': torch.get_rng_state(), 'cuda_rng': torch.cuda.get_rng_state_all() if device.type == 'cuda' else None, 'numpy_rng': np.random.get_state(), 'python_rng': random.getstate()}
        if improved:
            atomic_checkpoint({'model': model.state_dict(), 'config': asdict(config), 'epoch': epoch}, output_dir / 'best.pt')
        atomic_checkpoint(state, last)
        pd.DataFrame(history).to_csv(output_dir / 'history.csv', index=False)
        model.eval()
        with torch.no_grad():
            reconstruction_panel(fixed, model(fixed)[0], output_dir / 'reconstructions_latest.png', input_label='Original' if config.preprocessing == 'raw' else 'Model input: ' + config.preprocessing)
        print(json.dumps(row), flush=True)
        record_event('epoch_completed', **row)
        if stale >= config.patience:
            break
    (output_dir / 'training_status.json').write_text(json.dumps({'status': 'smoke_complete' if config.max_batches else 'complete', 'epochs_completed': len(history), 'best_validation_loss': best}, indent=2))
    record_event('completed', epochs_completed=len(history), best_validation_loss=best)
    return (model, pd.DataFrame(history))

def parse_bool(value):
    if isinstance(value, bool):
        return value
    if value.lower() in ('true', '1', 'yes'):
        return True
    if value.lower() in ('false', '0', 'no'):
        return False
    raise argparse.ArgumentTypeError('Expected true or false')

def build_parser():
    parser = argparse.ArgumentParser()
    parser.add_argument('--data', default='data')
    parser.add_argument('--out', required=True)
    parser.add_argument('--resume', action='store_true')
    for key, value in asdict(TrainConfig()).items():
        flag = '--' + key.replace('_', '-')
        if isinstance(value, bool):
            group = parser.add_mutually_exclusive_group()
            group.add_argument(flag, dest=key, type=parse_bool, nargs='?', const=True, default=value, metavar='BOOL')
            group.add_argument('--no-' + key.replace('_', '-'), dest=key, action='store_false')
        else:
            parser.add_argument(flag, type=type(value), default=value)
    return parser


In [ ]:
MAIN_RUNS = [{'name': 'v1',
  'version': 'v1',
  'latent_dim': 128,
  'epochs': 15,
  'batch_size': 16,
  'learning_rate': 0.0003,
  'structural_weight': 0.0,
  'gradient_weight': 0.0,
  'seed': 2026,
  'split_seed': 2026,
  'preprocessing': 'raw',
  'workers': 0,
  'patience': 5,
  'max_batches': 0,
  'augment': False},
 {'name': 'v2',
  'version': 'v2',
  'latent_dim': 128,
  'epochs': 15,
  'batch_size': 16,
  'learning_rate': 0.0003,
  'structural_weight': 0.1,
  'gradient_weight': 0.0,
  'seed': 2026,
  'split_seed': 2026,
  'preprocessing': 'raw',
  'workers': 0,
  'patience': 5,
  'max_batches': 0,
  'augment': False},
 {'name': 'v3',
  'version': 'v3',
  'latent_dim': 256,
  'epochs': 15,
  'batch_size': 16,
  'learning_rate': 0.0003,
  'structural_weight': 0.1,
  'gradient_weight': 0.0,
  'seed': 2026,
  'split_seed': 2026,
  'preprocessing': 'raw',
  'workers': 0,
  'patience': 5,
  'max_batches': 0,
  'augment': False},
 {'name': 'v4',
  'version': 'v4',
  'latent_dim': 256,
  'epochs': 15,
  'batch_size': 16,
  'learning_rate': 0.0003,
  'structural_weight': 0.1,
  'gradient_weight': 0.0,
  'seed': 2026,
  'split_seed': 2026,
  'preprocessing': 'contrast',
  'workers': 0,
  'patience': 5,
  'max_batches': 0,
  'augment': False},
 {'name': 'v5',
  'version': 'v5',
  'latent_dim': 256,
  'epochs': 15,
  'batch_size': 16,
  'learning_rate': 0.0003,
  'structural_weight': 0.1,
  'gradient_weight': 0.0,
  'seed': 2026,
  'split_seed': 2026,
  'preprocessing': 'footprint',
  'workers': 0,
  'patience': 5,
  'max_batches': 0,
  'augment': False},
 {'name': 'v6',
  'version': 'v6',
  'latent_dim': 256,
  'epochs': 15,
  'batch_size': 16,
  'learning_rate': 0.0003,
  'structural_weight': 0.1,
  'gradient_weight': 0.0,
  'seed': 2026,
  'split_seed': 2026,
  'preprocessing': 'border_fill',
  'workers': 0,
  'patience': 5,
  'max_batches': 0,
  'augment': False},
 {'name': 'v7',
  'version': 'v7',
  'latent_dim': 256,
  'epochs': 15,
  'batch_size': 16,
  'learning_rate': 0.0003,
  'structural_weight': 0.1,
  'gradient_weight': 0.1,
  'seed': 2026,
  'split_seed': 2026,
  'preprocessing': 'raw',
  'workers': 0,
  'patience': 5,
  'max_batches': 0,
  'augment': False},
 {'name': 'v8',
  'version': 'v8',
  'latent_dim': 256,
  'epochs': 15,
  'batch_size': 16,
  'learning_rate': 0.0003,
  'structural_weight': 0.1,
  'gradient_weight': 0.1,
  'seed': 2026,
  'split_seed': 2026,
  'preprocessing': 'raw',
  'workers': 0,
  'patience': 5,
  'max_batches': 0,
  'augment': True}]
ROBUSTNESS_RUNS = [{'name': 'robust_seed',
  'version': 'robust_seed',
  'latent_dim': 256,
  'epochs': 15,
  'batch_size': 16,
  'learning_rate': 0.0003,
  'structural_weight': 0.1,
  'gradient_weight': 0.0,
  'seed': 2027,
  'split_seed': 2026,
  'preprocessing': 'raw',
  'workers': 0,
  'patience': 5,
  'max_batches': 0,
  'augment': False},
 {'name': 'robust_split',
  'version': 'robust_split',
  'latent_dim': 256,
  'epochs': 15,
  'batch_size': 16,
  'learning_rate': 0.0003,
  'structural_weight': 0.1,
  'gradient_weight': 0.0,
  'seed': 2026,
  'split_seed': 2027,
  'preprocessing': 'raw',
  'workers': 0,
  'patience': 5,
  'max_batches': 0,
  'augment': False},
 {'name': 'improved_seed',
  'version': 'improved_seed',
  'latent_dim': 256,
  'epochs': 15,
  'batch_size': 16,
  'learning_rate': 0.0003,
  'structural_weight': 0.1,
  'gradient_weight': 0.0,
  'seed': 2027,
  'split_seed': 2026,
  'preprocessing': 'contrast',
  'workers': 0,
  'patience': 5,
  'max_batches': 0,
  'augment': False},
 {'name': 'improved_split',
  'version': 'improved_split',
  'latent_dim': 256,
  'epochs': 15,
  'batch_size': 16,
  'learning_rate': 0.0003,
  'structural_weight': 0.1,
  'gradient_weight': 0.0,
  'seed': 2026,
  'split_seed': 2027,
  'preprocessing': 'contrast',
  'workers': 0,
  'patience': 5,
  'max_batches': 0,
  'augment': False},
 {'name': 'v7_robust_seed',
  'version': 'v7_robust_seed',
  'latent_dim': 256,
  'epochs': 15,
  'batch_size': 16,
  'learning_rate': 0.0003,
  'structural_weight': 0.1,
  'gradient_weight': 0.1,
  'seed': 2027,
  'split_seed': 2026,
  'preprocessing': 'raw',
  'workers': 0,
  'patience': 5,
  'max_batches': 0,
  'augment': False},
 {'name': 'v7_robust_split',
  'version': 'v7_robust_split',
  'latent_dim': 256,
  'epochs': 15,
  'batch_size': 16,
  'learning_rate': 0.0003,
  'structural_weight': 0.1,
  'gradient_weight': 0.1,
  'seed': 2026,
  'split_seed': 2027,
  'preprocessing': 'raw',
  'workers': 0,
  'patience': 5,
  'max_batches': 0,
  'augment': False}]
RUN_ROBUSTNESS = False  # Enable to train missing independent repeats on a fresh runtime.
run_specs = MAIN_RUNS + (ROBUSTNESS_RUNS if RUN_ROBUSTNESS else [])
for spec in run_specs:
    run = ROOT / 'outputs' / spec['name']
    config = TrainConfig(**{k:v for k,v in spec.items() if k != 'name'})
    ensure_training(DATA,run,config)
    history = pd.read_csv(run/'history.csv')
    best = history.loc[history.validation_loss.idxmin()]
    print(spec['name'],'| domain:',config.preprocessing,'| gradient:',config.gradient_weight,
          '| augment:',config.augment,'| epoch:',int(best.epoch),'| MSE:',best.validation_mse)
# Also validate any optional cached runs; never silently reuse a different configuration.
for spec in ROBUSTNESS_RUNS:
    run = ROOT/'outputs'/spec['name']
    if (run/'training_status.json').exists():
        ensure_training(DATA,run,TrainConfig(**{k:v for k,v in spec.items() if k != 'name'}))


## What I changed
| Run | Change | Reason |
|---|---|---|
| v1 | MSE and 128-dimensional vector | Establish the baseline |
| v2 | Add SSIM | Investigate smoothed texture |
| v3 | Increase the vector to 256 | Test additional capacity |
| v4 | Standardize brightness and contrast | Investigate the bright-candidate concentration |
| v5 | Add border-connected-zero treatment | Isolate the additional footprint effect |
| v6 | Near-black mask and nearest-value fill | Test remaining border fragments |
| v7 | Add 0.1 gradient loss | Test edge reconstruction |
| v8 | Add flips and quarter-turns to v7 | Test orientation sensitivity |

The executed v7/v8 loss is 0.9 MSE + 0.1 (1-SSIM) + 0.1 Gradient. Gradient means adjacent-pixel finite differences, not Sobel filtering. The original v7 protocol described 0.8 MSE; I preserve and disclose that deviation.

The standardized experiments reconstruct different input values. Their MSE is not directly comparable with raw-image v3 MSE.


In [ ]:
fig,axes = plt.subplots(2,2,figsize=(12,7))
for row,names in enumerate([['v1','v2','v3','v7','v8'],['v4','v5','v6']]):
    for name in names:
        h = pd.read_csv(ROOT/'outputs'/name/'history.csv')
        axes[row,0].plot(h.epoch,h.validation_mse,label=name)
        axes[row,1].plot(h.epoch,h.validation_ssim,label=name)
    for col,metric in enumerate(['MSE','SSIM']):
        axes[row,col].set(xlabel='Epoch',ylabel=metric,
                          title='Raw input' if row == 0 else 'Standardized inputs: different targets')
        axes[row,col].legend()
fig.tight_layout()
display(fig)
plt.close(fig)
for name in ['v3','v4','v5']:
    print(name)
    display(DisplayImage(filename=str(ROOT/'outputs'/name/'reconstructions_latest.png')))


## Phase 2.1 — Isolation Forest on the latent vector
I fit 2,000 trees with 256 samples per tree, only on training latents. My earlier controlled comparison found better forest-seed agreement than at 400 trees. Novelty is negative score_samples: larger means more unusual. I ignore the library's contamination cutoff.

## Phase 2.2 — My statistical boundary
The original adjusted-boxplot threshold exceeded every v1 score. Its histogram was multimodal, so I investigated a distribution model instead of lowering the threshold to obtain five examples.

I fit one to four Gaussian components on calibration scores, choose minimum BIC, and use **T = max(component mean + 3 × component standard deviation)**. The rule puts a boundary beyond each fitted population rather than labelling the entire high-score population anomalous.

I refit model selection in 100 whole-source bootstrap samples. This is conditional uncertainty, not a guaranteed false-positive rate. Correlated crops, unknown contamination and limited model selection remain weaknesses. The same rule applies to every run; zero candidates is allowed.


In [ ]:
import numpy as np
from statsmodels.stats.stattools import medcouple

def mixture_fence(scores, seed=2026):
    """Envelope beyond every fitted score population; no contamination fraction."""
    from sklearn.mixture import GaussianMixture
    from threadpoolctl import threadpool_limits
    from scipy.stats import norm
    values = np.asarray(scores, dtype=float)
    if values.ndim != 1 or len(values) < 30 or (not np.isfinite(values).all()) or (values.std() < 1e-08):
        raise ValueError('Need at least 30 finite, nonconstant scores for a mixture')
    with threadpool_limits(limits=2):
        models = [GaussianMixture(n_components=k, n_init=5, random_state=seed, reg_covar=1e-06).fit(values[:, None]) for k in range(1, 5)]
    valid = [m for m in models if m.converged_]
    if not valid:
        raise ValueError('No mixture fit converged')
    best = min(valid, key=lambda m: m.bic(values[:, None]))
    means = best.means_.ravel()
    sigmas = np.sqrt(best.covariances_.ravel())
    threshold = float(np.max(means + 3 * sigmas))
    ordered = np.sort(values)
    cdf = np.sum(best.weights_[None, :] * norm.cdf((ordered[:, None] - means) / sigmas), axis=1)
    n = len(values)
    ks = float(max(np.max(np.arange(1, n + 1) / n - cdf), np.max(cdf - np.arange(n) / n)))
    return {'threshold': threshold, 'method': 'mixture_three_sigma', 'components': best.n_components, 'bic': {str(m.n_components): float(m.bic(values[:, None])) for m in valid}, 'weights': best.weights_.tolist(), 'means': means.tolist(), 'std': sigmas.tolist(), 'calibration_cdf_max_distance': ks, 'fitted_tail_mass': float(np.sum(best.weights_ * norm.sf((threshold - means) / sigmas))), 'component_count_at_search_limit': best.n_components == 4}

def upper_fence(scores):
    values = np.asarray(scores, dtype=float)
    if values.ndim != 1 or len(values) < 8 or (not np.isfinite(values).all()):
        raise ValueError('Need at least 8 finite calibration scores')
    q1, q3 = np.quantile(values, [0.25, 0.75])
    iqr = q3 - q1
    if iqr <= np.finfo(float).eps:
        raise ValueError('Degenerate calibration IQR; investigate representation and scores')
    mc = float(medcouple(values))
    multiplier = 3 if mc >= 0 else 4
    fence = q3 + 1.5 * np.exp(multiplier * mc) * iqr
    return {'threshold': float(fence), 'q1': float(q1), 'q3': float(q3), 'iqr': float(iqr), 'medcouple': mc}

def calibrate(scores, groups, bootstrap=100, seed=2026, method='adjusted_boxplot'):
    values, groups = (np.asarray(scores), np.asarray(groups))
    if values.shape != groups.shape:
        raise ValueError('Scores and groups must have equal shape')
    unique = np.unique(groups)
    if len(unique) < 8:
        raise ValueError('Too few calibration source groups for bootstrap')
    if method not in ('adjusted_boxplot', 'mixture_three_sigma'):
        raise ValueError('Unknown threshold method')
    boundary = upper_fence if method == 'adjusted_boxplot' else lambda x: mixture_fence(x, seed)
    result = boundary(values)
    result['method'] = method
    rng = np.random.default_rng(seed)
    indices = [np.flatnonzero(groups == g) for g in unique]
    thresholds = []
    for _ in range(bootstrap):
        sample = np.concatenate([indices[i] for i in rng.integers(0, len(unique), len(unique))])
        try:
            thresholds.append(boundary(values[sample])['threshold'])
        except ValueError:
            continue
    if len(thresholds) < max(10, bootstrap * 0.8):
        raise ValueError('Too many degenerate bootstrap replicates')
    median = np.median(values)
    mad = np.median(np.abs(values - median))
    q1, q3 = np.quantile(values, [0.25, 0.75])
    result.update({'bootstrap_interval_95': np.quantile(thresholds, [0.025, 0.975]).tolist(), 'bootstrap_thresholds': thresholds, 'bootstrap_valid': len(thresholds), 'calibration_images': len(values), 'calibration_sources': len(unique), 'tukey_comparison': float(q3 + 1.5 * (q3 - q1)), 'mad_comparison': float(median + 3 * 1.4826 * mad), 'interpretation': 'Descriptive distribution-based screening boundary; no nominal false-positive or false-discovery guarantee.'})
    return result

def select_flagged(frame, threshold, limit=5):
    """The limit is only for explanations, never for threshold determination."""
    return frame.loc[frame.novelty_score > threshold].sort_values(['novelty_score', 'filename'], ascending=[False, True]).head(limit)


In [ ]:
import argparse
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.stats import spearmanr

def evaluate(data_dir, run_dir, bootstrap=100, projection=True, allow_smoke=False, method='adjusted_boxplot', trees=400):
    run_dir = Path(run_dir)
    checkpoint = torch.load(run_dir / 'best.pt', map_location='cpu', weights_only=False)
    config = checkpoint['config']
    if method != 'adjusted_boxplot' or trees != 400:
        run_dir = run_dir / (method + (f'_trees{trees}' if trees != 400 else ''))
        run_dir.mkdir(parents=True, exist_ok=True)
    if config['max_batches'] and (not allow_smoke):
        raise ValueError('Smoke checkpoint cannot produce competition results')
    torch.set_num_threads(4)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = ConvAutoencoder(config['latent_dim']).to(device)
    model.load_state_dict(checkpoint['model'])
    model.eval()
    frame = load_manifest(data_dir, config.get('split_seed', config['seed']))
    preprocessing = config.get('preprocessing', 'raw')
    loader = DataLoader(CropDataset(frame, preprocessing), batch_size=config['batch_size'], num_workers=0)
    latents, errors = ([], [])
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            reconstruction, z = model(x)
            latents.append(z.cpu().numpy())
            errors.extend((x - reconstruction).square().mean((1, 2, 3)).cpu().tolist())
    z = np.concatenate(latents)
    np.save(run_dir / 'latents.npy', z)
    np.save(run_dir / 'latent_filenames.npy', frame.filename.to_numpy(dtype=str))
    train = frame.split.eq('train').to_numpy()
    calibration_mask = frame.split.eq('calibration').to_numpy()
    forest = IsolationForest(n_estimators=trees, max_samples=256, contamination='auto', random_state=config['seed'], n_jobs=2)
    forest.fit(z[train])
    frame['novelty_score'] = -forest.score_samples(z)
    frame['reconstruction_mse'] = errors
    calibration = calibrate(frame.loc[calibration_mask, 'novelty_score'], frame.loc[calibration_mask, 'source_image_id'], bootstrap, config['seed'], method)
    frame['flagged'] = frame.novelty_score > calibration['threshold']
    boot = np.asarray(calibration['bootstrap_thresholds'])
    frame['threshold_bootstrap_flag_frequency'] = (frame.novelty_score.to_numpy()[:, None] > boot[None, :]).mean(1)
    stability = []
    for seed in (config['seed'] + 1, config['seed'] + 2):
        other = IsolationForest(n_estimators=trees, max_samples=256, contamination='auto', random_state=seed, n_jobs=2).fit(z[train])
        other_scores = -other.score_samples(z)
        boundary = (upper_fence(other_scores[calibration_mask]) if method == 'adjusted_boxplot' else mixture_fence(other_scores[calibration_mask], config['seed']))['threshold']
        other_flags = other_scores > boundary
        primary_flags = frame.flagged.to_numpy()
        union = np.count_nonzero(primary_flags | other_flags)
        stability.append({'seed': seed, 'spearman': float(spearmanr(frame.novelty_score, other_scores).statistic), 'flag_count': int(other_flags.sum()), 'threshold': boundary, 'flag_jaccard': float(np.count_nonzero(primary_flags & other_flags) / union) if union else None})
    covariance = np.cov(z[train], rowvar=False)
    eigenvalues = np.linalg.eigvalsh(covariance).clip(0)
    proportions = eigenvalues / max(eigenvalues.sum(), 1e-12)
    effective_rank = float(np.exp(-np.sum(proportions * np.log(proportions + 1e-12))))
    diagnostics = {'effective_rank': effective_rank, 'latent_dimensions': z.shape[1], 'latent_variance': z[train].var(0).tolist(), 'forest_seed_stability': stability, 'flagged_total': int(frame.flagged.sum()), 'flagged_by_split': frame.groupby('split').flagged.agg(['sum', 'count', 'mean']).to_dict(), 'smoke_checkpoint': bool(config['max_batches']), 'threshold_exceeds_maximum': bool(calibration['threshold'] > frame.novelty_score.max())}
    diagnostics['n_estimators'] = trees
    diagnostics['preprocessing'] = preprocessing
    diagnostics['reconstruction_domain'] = 'raw intensity / 255' if preprocessing == 'raw' else preprocessing + ' standardized input'
    if method == 'mixture_three_sigma':
        from scipy.stats import norm
        means = np.array(calibration['means'])
        std = np.array(calibration['std'])
        weights = np.array(calibration['weights'])
        heldout = np.sort(frame.loc[frame.split.eq('validation'), 'novelty_score'])
        cdf = np.sum(weights * norm.cdf((heldout[:, None] - means) / std), axis=1)
        n = len(heldout)
        diagnostics['validation_cdf_max_distance'] = float(max(np.max(np.arange(1, n + 1) / n - cdf), np.max(cdf - np.arange(n) / n)))
        diagnostics['validation_cdf_note'] = 'Descriptive held-out density-fit diagnostic; correlated sources preclude an ordinary IID KS p-value.'
    (run_dir / 'calibration.json').write_text(json.dumps(calibration, indent=2))
    (run_dir / 'diagnostics.json').write_text(json.dumps(diagnostics, indent=2))
    frame.drop(columns='path').to_csv(run_dir / 'novelty_scores.csv', index=False)
    joblib.dump(forest, run_dir / 'isolation_forest.joblib')
    score_plots(frame, calibration, run_dir / 'score_distribution.png')
    source_summary = frame.groupby('source_image_id').agg(crops=('filename', 'size'), flagged=('flagged', 'sum'), mean_novelty=('novelty_score', 'mean'), latitude=('latitude', 'first'), longitude=('longitude', 'first'), sun_angle=('sun_angle', 'first'), season=('season', 'first'), resolution=('resolution', 'first'))
    source_summary['flag_rate'] = source_summary.flagged / source_summary.crops
    source_summary.to_csv(run_dir / 'source_summary.csv')
    for column in ('season', 'resolution', 'longitude'):
        crop_summary = frame.groupby(column).flagged.agg(['count', 'sum', 'mean'])
        equal_source = source_summary.groupby(column).flag_rate.mean()
        crop_summary['equal_source_mean_flag_rate'] = equal_source
        crop_summary.to_csv(run_dir / f'metadata_{column}.csv')
    selected = select_flagged(frame, calibration['threshold'])
    selected.drop(columns='path').to_csv(run_dir / 'selected_for_interpretation.csv', index=False)
    for _, row in selected.iterrows():
        x, _ = CropDataset(pd.DataFrame([row]), preprocessing)[0]
        with torch.no_grad():
            rec = model(x[None].to(device))[0][0, 0].cpu().numpy()
        if preprocessing == 'raw':
            heatmap_panel(x[0].numpy(), rec, row, calibration['threshold'], run_dir / f'heatmap_{Path(row.filename).stem}.png')
        else:
            raw, _ = CropDataset(pd.DataFrame([row]))[0]
            standardized_heatmap_panel(raw[0].numpy(), x[0].numpy(), rec, row, calibration['threshold'], run_dir / f'heatmap_{Path(row.filename).stem}.png', preprocessing)
    if projection:
        import matplotlib.pyplot as plt
        scale = z[train].std(0).clip(1e-06)
        standardized = (z - z[train].mean(0)) / scale
        reduced = PCA(n_components=min(50, z.shape[1]), random_state=config['seed']).fit_transform(standardized)
        for perplexity in (30, 70):
            xy = TSNE(n_components=2, perplexity=perplexity, init='pca', learning_rate='auto', random_state=config['seed'], max_iter=1000).fit_transform(reduced)
            pd.DataFrame({'filename': frame.filename, 'x': xy[:, 0], 'y': xy[:, 1]}).to_csv(run_dir / f'tsne_{perplexity}.csv', index=False)
            fig, axes = plt.subplots(1, 3, figsize=(15, 4))
            for ax, values, title in zip(axes, [frame.novelty_score, frame.sun_angle, pd.Categorical(frame.source_image_id).codes], ['Novelty score', 'Sun angle as supplied', 'Source ID (categorical colors)']):
                points = ax.scatter(xy[:, 0], xy[:, 1], c=values, s=3, cmap='viridis', rasterized=True)
                fig.colorbar(points, ax=ax)
                ax.set_title(title)
                ax.set_xticks([])
                ax.set_yticks([])
            fig.suptitle(f't-SNE perplexity {perplexity}; qualitative projection, not geological labels')
            fig.tight_layout()
            fig.savefig(run_dir / f'tsne_{perplexity}.png', dpi=160)
            plt.close(fig)
    print(json.dumps({k: v for k, v in diagnostics.items() if k != 'latent_variance'}, indent=2), flush=True)
    return (frame, calibration, diagnostics)


In [ ]:
analysis_name = 'mixture_three_sigma_trees2000'
for spec in MAIN_RUNS + ROBUSTNESS_RUNS:
    run = ROOT/'outputs'/spec['name']
    if not (run/'training_status.json').exists():
        print(spec['name'],': optional repeat absent from this runtime')
        continue
    analysis = run/analysis_name
    if not (analysis/'selected_for_interpretation.csv').exists():
        evaluate(DATA,run,bootstrap=100,projection=False,
                 method='mixture_three_sigma',trees=2000)
    d = json.loads((analysis/'diagnostics.json').read_text())
    c = json.loads((analysis/'calibration.json').read_text())
    print(spec['name'],'| threshold:',round(c['threshold'],6),'| flags:',d['flagged_total'])


## My selection after the checks

I retain v3 as the screening reference. v7 improves validation MSE by 1.68% and gradient error by 0.52%, but four of its five selected crops remain extremely bright and forest-repeat agreement weakens. v4-v6 reduce specific nuisance responses while introducing border or fill concerns; v6 fails its prespecified promotion criteria. v8 has zero image-only flags. No tested replacement establishes better hidden-anomaly detection. v3 itself remains brightness-confounded and unstable across independent runs; its flags are investigation candidates, not confirmed anomalies.

For v4, changing initialization gives flag-set Jaccard 0.207 and full-ranking correlation 0.867; changing the source split gives 0.360 and 0.866. The three contrast runs flag 24, 11 and 10 crops, with 4 in their intersection. On 1,253 crops held out from both source partitions, rank correlation is 0.897. For raw v3, the corresponding initialization/split Jaccards remain 0.150/0.294, with zero crops flagged in all three raw runs. These are limited sensitivity checks, not detection accuracy. Initialization repeats change both encoder and forest seeds; separate forest-only repeats isolate detector randomness.

In [ ]:
RECORDED_SELECTION = {'run': 'v3',
 'analysis': 'mixture_three_sigma_trees2000',
 'status': 'Complete',
 'recorded_at': '2026-09-17T17:03:33+05:30'}
RUN = ROOT/'outputs'/RECORDED_SELECTION['run']
ANALYSIS = RUN/analysis_name
scored = pd.read_csv(ANALYSIS/'novelty_scores.csv')
calibration = json.loads((ANALYSIS/'calibration.json').read_text())
diagnostics = json.loads((ANALYSIS/'diagnostics.json').read_text())
display({k:v for k,v in calibration.items() if k != 'bootstrap_thresholds'})
display(scored.groupby('split').flagged.agg(['count','sum','mean']))
display(DisplayImage(filename=str(ANALYSIS/'score_distribution.png')))


## Phase 1.3 — Looking at the latent space
PCA and t-SNE are qualitative diagnostics only; the Isolation Forest receives the full 256-dimensional vector, not the projection. I compare two perplexities (30 and 70) and colour each point by its novelty score, its sun angle and its source image ID.

**What I observe:** At both perplexities, the projection shows a primary dense region with a sparse fringe of higher-novelty points. The broad cloud structure does not resolve into clearly separated terrain classes, which is expected: this is an unsupervised representation trained to reconstruct rather than to cluster geology. A few elongated threads near the fringe correspond to crops that score most anomalously; these are the candidates the Isolation Forest assigns its highest novelty scores.

**Effective rank interpretation:** The effective rank is approximately 31 out of 256 dimensions. This means roughly 31 dimensions carry most of the variance — a moderate compression without complete collapse. It does not mean 31 geological classes exist, nor does it confirm encoder collapse (a structureless single blob). The between-source variance fraction is approximately 0.23, meaning source-level acquisition effects account for about a quarter of the total latent spread.

**Limitation:** A visually distinguishable fringe in t-SNE does not confirm that those points are genuine anomalies. t-SNE can distort global distances; the perplexity controls local vs. global neighbourhood emphasis. These projections are presented as visual diagnostics, not proof of geological clustering.


In [ ]:
"""Generate t-SNE diagnostics from existing latents without rerunning calibration."""
import argparse
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from threadpoolctl import threadpool_limits
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def project(directory):
    z = np.load(directory / 'latents.npy')
    frame = pd.read_csv(directory / 'novelty_scores.csv')
    assert np.array_equal(np.load(directory / 'latent_filenames.npy'), frame.filename)
    train = frame.split.eq('train').to_numpy()
    scaled = (z - z[train].mean(0)) / z[train].std(0).clip(1e-06)
    with threadpool_limits(limits=2):
        reduced = PCA(n_components=min(50, z.shape[1]), random_state=2026).fit_transform(scaled)
        for perplexity in [30, 70]:
            xy = TSNE(perplexity=perplexity, max_iter=1000, init='pca', learning_rate='auto', random_state=2026, n_jobs=2).fit_transform(reduced)
            pd.DataFrame({'filename': frame.filename, 'x': xy[:, 0], 'y': xy[:, 1]}).to_csv(directory / f'tsne_{perplexity}.csv', index=False)
            fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
            for ax, values, label in zip(axes[:2], [frame.novelty_score, frame.sun_angle], ['Novelty score', 'Sun angle as supplied']):
                points = ax.scatter(xy[:, 0], xy[:, 1], c=values, s=3, cmap='viridis', rasterized=True)
                fig.colorbar(points, ax=ax, shrink=0.8)
                ax.set_title(label)
            largest = frame.source_image_id.value_counts().head(5).index
            axes[2].scatter(xy[:, 0], xy[:, 1], s=2, c='lightgray', label='Other sources')
            for i, source in enumerate(largest):
                mask = frame.source_image_id.eq(source)
                axes[2].scatter(xy[mask, 0], xy[mask, 1], s=4, color=plt.get_cmap('tab10')(i), label=source)
            axes[2].legend(fontsize=7, loc='best')
            axes[2].set_title('Five largest source groups')
            for ax in axes:
                ax.set_xticks([])
                ax.set_yticks([])
            fig.suptitle(f't-SNE, perplexity {perplexity}: qualitative diagnostic, not confirmed terrain classes')
            fig.tight_layout()
            fig.savefig(directory / f'tsne_{perplexity}.png', dpi=160)
            plt.close(fig)
            print(f'Completed t-SNE {perplexity}', flush=True)


In [ ]:
if not all((ANALYSIS/f'tsne_{p}.png').exists() for p in [30,70]):
    project(ANALYSIS)
for p in [30,70]:
    display(DisplayImage(filename=str(ANALYSIS/f'tsne_{p}.png')))
print('Effective rank:',diagnostics['effective_rank'],'of',diagnostics['latent_dimensions'])


## Did the improvements reduce the problem?
I report both full-ranking correlation and overlap of the flagged sets. A high correlation can conceal disagreement in the extreme tail. I also hold each fitted forest and cutoff fixed while applying 0.8 × x + 0.1 outside the inferred black border. Keeping the footprint fixed simulates exposure changes rather than changing crop support. This is a nuisance-sensitivity check, not labelled detection accuracy.


In [ ]:
"""Compare saved experiments without inventing hidden-label accuracy."""
from pathlib import Path
import argparse, json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def summarize(root):
    output = root / 'outputs' / 'comparison'
    output.mkdir(exist_ok=True)
    rows = []
    frames = {}
    for name in ['v1', 'v2', 'v3', 'robust_seed', 'robust_split', 'v4', 'v5', 'improved_seed', 'improved_split', 'v6', 'v7', 'v7_robust_seed', 'v7_robust_split', 'v8']:
        run = root / 'outputs' / name
        if not (run / 'training_status.json').exists():
            continue
        config = json.loads((run / 'config.json').read_text())
        history = pd.read_csv(run / 'history.csv')
        best = history.loc[history.validation_loss.idxmin()]
        row = {'run': name, 'latent_dim': config['latent_dim'], 'ssim_weight': config['structural_weight'], 'seed': config['seed'], 'split_seed': config.get('split_seed', config['seed']), 'best_epoch': int(best.epoch), 'validation_mse': best.validation_mse, 'validation_ssim': best.validation_ssim, 'validation_gradient': best.validation_gradient, 'training_seconds': history.seconds.sum()}
        row['input_domain'] = config.get('preprocessing', 'raw')
        row['gradient_weight'] = config.get('gradient_weight', 0.0)
        row['augment'] = config.get('augment', False)
        analysis = run / 'mixture_three_sigma_trees2000'
        if not (analysis / 'diagnostics.json').exists():
            analysis = run / 'mixture_three_sigma'
        if (analysis / 'diagnostics.json').exists():
            diagnostics = json.loads((analysis / 'diagnostics.json').read_text())
            calibration = json.loads((analysis / 'calibration.json').read_text())
            row['trees'] = diagnostics.get('n_estimators', 400)
            row.update({'effective_rank': diagnostics['effective_rank'], 'threshold': calibration['threshold'], 'threshold_low': calibration['bootstrap_interval_95'][0], 'threshold_high': calibration['bootstrap_interval_95'][1], 'flagged': diagnostics['flagged_total'], 'validation_cdf_distance': diagnostics['validation_cdf_max_distance'], 'components': calibration['components'], 'forest_min_spearman': min((x['spearman'] for x in diagnostics['forest_seed_stability']))})
            frames[name] = pd.read_csv(analysis / 'novelty_scores.csv').set_index('filename')
        rows.append(row)
    table = pd.DataFrame(rows)
    table.to_csv(output / 'experiments.csv', index=False)
    pairs = []
    for i, left in enumerate(frames):
        for right in list(frames)[i + 1:]:
            a, b = frames[left].align(frames[right], join='inner', axis=0)
            union = np.count_nonzero(a.flagged | b.flagged)
            heldout = (a.split != 'train') & (b.split != 'train')
            pairs.append({'left': left, 'right': right, 'spearman_all': float(spearmanr(a.novelty_score, b.novelty_score).statistic), 'flag_jaccard': float(np.count_nonzero(a.flagged & b.flagged) / union) if union else None, 'common_heldout_images': int(heldout.sum()), 'spearman_common_heldout': float(spearmanr(a.loc[heldout, 'novelty_score'], b.loc[heldout, 'novelty_score']).statistic)})
            heldout_union = np.count_nonzero((a.flagged | b.flagged) & heldout)
            pairs[-1].update(all_images=len(a), flag_union_all=int(union), flag_jaccard_common_heldout=float(np.count_nonzero(a.flagged & b.flagged & heldout) / heldout_union) if heldout_union else None, flag_union_common_heldout=int(heldout_union))
    pd.DataFrame(pairs).to_csv(output / 'run_stability.csv', index=False)
    statistics_path = root / 'outputs/audit/image_statistics.csv'
    stats = pd.read_csv(statistics_path).set_index('filename') if statistics_path.exists() else None
    confounds = []
    for name, frame in frames.items():
        if stats is None:
            continue
        joined = frame.join(stats)
        total = ((joined.novelty_score - joined.novelty_score.mean()) ** 2).sum()
        source_means = joined.groupby('source_image_id').novelty_score.transform('mean')
        explained = float(((source_means - joined.novelty_score.mean()) ** 2).sum() / total)
        row = {'run': name, 'source_between_group_variance_fraction': explained}
        for c in ['mean', 'std', 'zero_fraction', 'sun_angle', 'latitude']:
            row['spearman_' + c] = float(spearmanr(joined[c], joined.novelty_score).statistic)
        confounds.append(row)
    if confounds:
        pd.DataFrame(confounds).to_csv(output / 'confounds.csv', index=False)
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.7))
    for name in ['v1', 'v2', 'v3']:
        run = root / 'outputs' / name
        if not (run / 'history.csv').exists():
            continue
        history = pd.read_csv(run / 'history.csv')
        for ax, metric in zip(axes, ['mse', 'ssim', 'gradient']):
            ax.plot(history.epoch, history['validation_' + metric], label=name)
    for ax, metric in zip(axes, ['MSE (lower better)', 'SSIM (higher better)', 'Gradient error (lower better)']):
        ax.set(xlabel='Epoch', ylabel=metric)
        ax.legend()
    fig.tight_layout()
    fig.savefig(output / 'validation_curves.png', dpi=170)
    plt.close(fig)
    print(table.to_string(index=False))
    print(pd.DataFrame(pairs).to_string(index=False))
    print(pd.DataFrame(confounds).to_string(index=False))


In [ ]:
summarize(ROOT)
display(pd.read_csv(ROOT/'outputs/comparison/experiments.csv'))
pairs = pd.read_csv(ROOT/'outputs/comparison/run_stability.csv')
display(pairs[((pairs.left == 'v3') & pairs.right.isin(['robust_seed','robust_split'])) | ((pairs.left == 'v7') & pairs.right.isin(['v7_robust_seed','v7_robust_split']))])
print('flag_jaccard uses all crops; flag_jaccard_common_heldout uses only crops held out from both models. Empty union is undefined.')


In [ ]:
"""Hold the fitted forest/cutoff fixed while perturbing validation illumination."""
from pathlib import Path
import sys, argparse, json
import numpy as np
import pandas as pd
import torch, joblib

def measure(root, runs):
    torch.set_num_threads(2)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    rows = []
    for name in runs:
        run = root / 'outputs' / name
        analysis = run / 'mixture_three_sigma_trees2000'
        if not (analysis / 'diagnostics.json').exists():
            continue
        checkpoint = torch.load(run / 'best.pt', map_location='cpu', weights_only=False)
        config = checkpoint['config']
        mode = config.get('preprocessing', 'raw')
        frame = load_manifest(root / 'data', config.get('split_seed', config['seed']))
        frame = frame[frame.split == 'validation'].sample(n=min(512, int((frame.split == 'validation').sum())), random_state=42).reset_index(drop=True)
        model = ConvAutoencoder(config['latent_dim']).to(device)
        model.load_state_dict(checkpoint['model'])
        model.eval()
        original = []
        altered = []
        for x, _ in CropDataset(frame):
            a = x[0].numpy()
            mask = border_zero_mask(a)
            b = 0.8 * a + 0.1
            b[mask] = 0
            original.append(preprocess_array(a, mode))
            altered.append(preprocess_array(b, mode))

        def encode(arrays):
            outputs = []
            with torch.no_grad():
                for i in range(0, len(arrays), 16):
                    batch = torch.from_numpy(np.stack(arrays[i:i + 16])[:, None]).to(device)
                    outputs.append(model.encode(batch).cpu().numpy())
            return np.concatenate(outputs)
        z0, z1 = (encode(original), encode(altered))
        forest = joblib.load(analysis / 'isolation_forest.joblib')
        s0, s1 = (-forest.score_samples(z0), -forest.score_samples(z1))
        threshold = json.loads((analysis / 'calibration.json').read_text())['threshold']
        delta = np.abs(s1 - s0)
        result = {'run': name, 'preprocessing': mode, 'validation_crops': len(frame), 'gain': 0.8, 'offset': 0.1, 'median_absolute_score_change': float(np.median(delta)), 'p95_absolute_score_change': float(np.quantile(delta, 0.95)), 'mean_input_absolute_change': float(np.abs(np.array(original) - np.array(altered)).mean()), 'median_relative_latent_change': float(np.median(np.linalg.norm(z1 - z0, axis=1) / np.maximum(np.linalg.norm(z0, axis=1), 1e-08))), 'flag_flips': int(np.count_nonzero((s0 > threshold) != (s1 > threshold))), 'original_flags': int((s0 > threshold).sum()), 'perturbed_flags': int((s1 > threshold).sum())}
        rows.append(result)
        pd.DataFrame({'filename': frame.filename, 'original_score': s0, 'perturbed_score': s1, 'absolute_change': delta}).to_csv(analysis / 'photometric_sensitivity.csv', index=False)
    table = pd.DataFrame(rows)
    out = root / 'outputs/comparison'
    out.mkdir(exist_ok=True, parents=True)
    table.to_csv(out / 'photometric_sensitivity.csv', index=False)
    print(table.to_string(index=False))


In [ ]:
measure(ROOT,['v3','v4','v5'])
display(pd.read_csv(ROOT/'outputs/comparison/photometric_sensitivity.csv'))


### What the brightness test misses
The illumination perturbation is deliberately simple and favors an affine-invariant transform. I do not treat passing it as proof that real anomalies are retained.

After calibration, I inspected the five leading v4 examples. All contain a conspicuous black strip or border. The exact-zero mask in v5 can leave near-black edge fragments, so filling zeros does not remove every acquisition feature. These are concrete reasons to avoid promoting preprocessing solely on its brightness result.


In [ ]:
for name in ['v4','v5']:
    print(name,': the same validation crop, shown after each independently calibrated selection')
    recorded_example = ROOT/'outputs'/name/analysis_name/'heatmap_sample_09852.png'
    if recorded_example.exists():
        display(DisplayImage(filename=str(recorded_example)))
    else:
        print('This saved example is absent from the new run; inspect its own selected crops.')


## Phase 2.3 — Supplied metadata and source context
I use the organizer's crop-to-source index only; I do not infer coordinates from appearance. Longitude has only two distinct values (0 and 180) so geographic comparison is limited to a binary split. Latitude has 24 coarse values spanning −90° to +90°. Resolution has three values (0.25, 0.5, 1.0) whose units are unspecified by the organizers. Sun-angle semantics (incidence vs. elevation) are not defined in the supplied metadata.

Crop-weighted and equal-source flag rates differ because source sizes range from 1 to 278 crops. I report both to avoid misrepresenting small-source observations.

**Key finding:** Three of the five selected crops come from a single source, SRC_154. Its equal-source flag rate is not exceptional, but SRC_154 contributes 3 of 5 candidates. This source concentration is the dominant geographic pattern in the selection — source acquisition conditions (illumination, sensor state) are a plausible driver of the elevated novelty scores, not necessarily geographically anomalous terrain. This limitation is disclosed in every candidate's physical hypothesis.

### Phase 2.4 — Optional metadata fusion
The primary score uses image latents only. I separately compare matched five-forest image-only and metadata-fused ensembles on the v8 latents. Each pipeline fits its own independent calibration boundary on its own calibration scores — the fused and image-only thresholds are not compared against each other. Metadata enters before the forest, not the decoder. Sun angle and resolution (2 numerical dimensions, training-scaled) plus four one-hot season dimensions add 6 coordinates to the 256-dim image latent, giving a 262-dimensional fused vector.

Both the image-only and fused ensembles produce zero flags under their respective mixture-envelope boundaries. The Spearman correlation between image-only and fused scores is 0.998, indicating that the six metadata dimensions add negligible independent signal at this scale. The candidate set is unchanged. This is the correct result to report — manufacturing candidates by lowering the threshold is explicitly prohibited by the rules.


In [ ]:
display(pd.read_csv(ANALYSIS/'source_summary.csv').sort_values('flag_rate',ascending=False).head(15))
for field in ['season','resolution','longitude']:
    print(field)
    display(pd.read_csv(ANALYSIS/f'metadata_{field}.csv'))


In [ ]:
"""Matched image-only/fused ensembles; optional diagnostic, not the primary detector."""
import sys, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from scipy.stats import spearmanr

def fusion(root=Path('.')):
    root = Path(root)
    analysis = root / 'outputs/v8/mixture_three_sigma_trees2000'
    out = root / 'outputs/v8/fusion_controlled'
    out.mkdir(parents=True, exist_ok=True)
    config = json.loads((root / 'outputs/v8/config.json').read_text())
    manifest = load_manifest(root / 'data', config.get('split_seed', config['seed']))
    names = np.load(analysis / 'latent_filenames.npy', allow_pickle=False)
    latents = np.load(analysis / 'latents.npy')
    frame = pd.DataFrame({'filename': names}).merge(manifest, on='filename', validate='one_to_one')
    assert len(frame) == len(latents) and (not frame.isna().any().any())
    train = frame.split.eq('train').to_numpy()
    cal = frame.split.eq('calibration').to_numpy()
    inputs = {str(p.relative_to(root)): hashlib.sha256(p.read_bytes()).hexdigest() for p in [analysis / 'latents.npy', analysis / 'latent_filenames.npy', root / 'data/source_image_metadata.csv', root / 'data/crop_metadata_index.csv', root / 'outputs/v8/config.json']}
    if (out / 'summary.json').exists():
        saved = json.loads((out / 'summary.json').read_text())
        portable = lambda d: {k.replace('\\', '/'): v for k, v in d.items()}
        if portable(saved['input_sha256']) != portable(inputs):
            raise ValueError('Fusion cache input mismatch; use a new output folder')
        for name in ['image_only', 'fused']:
            for suffix in ['scores.csv', 'calibration.json', 'detector.joblib']:
                if not (out / f'{name}_{suffix}').exists():
                    raise FileNotFoundError('Incomplete fusion cache')
        return saved
    scaler = StandardScaler().fit(frame.loc[train, ['sun_angle', 'resolution']])
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit(frame.loc[train, ['season']])
    metadata = np.concatenate([scaler.transform(frame[['sun_angle', 'resolution']]), encoder.transform(frame[['season']])], axis=1)
    vectors = {'image_only': latents, 'fused': np.concatenate([latents, metadata], axis=1)}
    scores = {}
    sets = {}
    thresholds = {}
    seeds = [2026, 2027, 2028, 2029, 2030]
    for name, z in vectors.items():
        forests = []
        values = []
        for seed in seeds:
            forest = IsolationForest(n_estimators=2000, max_samples=256, contamination='auto', random_state=seed, n_jobs=2).fit(z[train])
            values.append(-forest.score_samples(z))
            forests.append(forest)
        score = np.mean(values, axis=0)
        boundary = mixture_fence(score[cal], seed=2026)
        flags = score > boundary['threshold']
        scores[name] = score
        sets[name] = set(frame.loc[flags, 'filename'])
        thresholds[name] = boundary['threshold']
        result = frame.drop(columns='path').copy()
        result['novelty_score'] = score
        result['flagged'] = flags
        result.to_csv(out / f'{name}_scores.csv', index=False)
        (out / f'{name}_calibration.json').write_text(json.dumps(boundary, indent=2))
        joblib.dump({'forests': forests, 'scaler': scaler if name == 'fused' else None, 'encoder': encoder if name == 'fused' else None, 'numerical_columns': ['sun_angle', 'resolution'], 'categorical_columns': ['season'], 'threshold': boundary['threshold'], 'seeds': seeds, 'input_sha256': inputs}, out / f'{name}_detector.joblib', compress=3)
        print(name, 'matched ensemble flags:', int(flags.sum()), flush=True)
        del forests, forest
    union = sets['image_only'] | sets['fused']
    summary = {'run': 'v8', 'seeds': seeds, 'n_estimators_per_forest': 2000, 'dimensions': {n: z.shape[1] for n, z in vectors.items()}, 'thresholds': thresholds, 'flags': {n: len(s) for n, s in sets.items()}, 'flag_jaccard': len(sets['image_only'] & sets['fused']) / len(union) if union else None, 'added_by_fusion': sorted(sets['fused'] - sets['image_only']), 'removed_by_fusion': sorted(sets['image_only'] - sets['fused']), 'spearman_all': float(spearmanr(scores['image_only'], scores['fused']).statistic), 'input_sha256': inputs, 'interpretation': 'Matched ensemble diagnostic; each boundary is fitted on its own calibration scores. Empty sets do not establish accuracy. Metadata enters only before the forest, not the decoder. No bonus score is claimed.'}
    (out / 'summary.json').write_text(json.dumps(summary, indent=2))
    return summary


In [ ]:
fusion_result = fusion(ROOT)
display({k:v for k,v in fusion_result.items() if k != 'input_sha256'})
print('An empty flag union has undefined Jaccard; it does not demonstrate correct detection.')


## Later ablations: what failed
v7 lowers validation MSE by 1.68%, but four of its five selected images remain above the 99.8th brightness percentile. The darker selected crop also has conspicuous borders. I cannot claim that brightness bias was solved.

v6 reduces border-strip response, but fails the predeclared defect-response and forest-stability criteria. Its nearest-value fill introduces artificial stripes. These are negative results, not grounds for changing the threshold.


In [ ]:
for name in ['v6','v7']:
    folder = ROOT/'outputs'/name/analysis_name
    candidates = pd.read_csv(folder/'selected_for_interpretation.csv')
    print(name, 'selected candidates')
    display(candidates[['filename','novelty_score','source_image_id']])
    if len(candidates):
        display(DisplayImage(filename=str(folder/f'heatmap_{Path(candidates.iloc[0].filename).stem}.png')))


In [ ]:
"""Fixed, held-out response diagnostics; synthetic defects are never training labels."""
from pathlib import Path
from io import BytesIO
import sys, json
import numpy as np
import pandas as pd
from PIL import Image
import torch, joblib
BORDER_ANALYSIS_NAME = 'mixture_three_sigma_trees2000'

def jpeg(array):
    stream = BytesIO()
    Image.fromarray(np.rint(array * 255).astype(np.uint8)).save(stream, format='JPEG', quality=90)
    stream.seek(0)
    with Image.open(stream) as decoded:
        return np.array(decoded, dtype=np.float32) / 255

def variants(array):
    strip = array.copy()
    strip[:, :8] = 2 / 255
    bright = array.copy()
    bright[101:125, 101:125] = np.clip(bright[101:125, 101:125] + 0.35, 0, 1)
    dark = array.copy()
    dark[101:125, 101:125] = np.clip(dark[101:125, 101:125] - 0.35, 0, 1)
    checker = array.copy()
    yy, xx = np.indices((24, 24))
    checker[101:125, 101:125] = 0.1 + 0.8 * ((yy // 4 + xx // 4) % 2)
    return {'original': array, 'jpeg_control': jpeg(array), 'border_strip': jpeg(strip), 'bright_square': bright, 'dark_square': dark, 'checker': checker}

def border_response_checks():
    torch.set_num_threads(2)
    out = ROOT / 'outputs/border_experiment'
    out.mkdir(parents=True, exist_ok=True)
    manifest = load_manifest(ROOT / 'data', 2026)
    sample = manifest[manifest.split == 'validation'].sample(256, random_state=73).reset_index(drop=True)
    sample.drop(columns='path').to_csv(out / 'stress_sample.csv', index=False)
    arrays = []
    for path in sample.path:
        with Image.open(path) as im:
            arrays.append(np.array(im, dtype=np.float32) / 255)
    transformed = [variants(a) for a in arrays]
    rows = []
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    for name in ['v3', 'v5', 'v6']:
        run = ROOT / 'outputs' / name
        analysis = run / BORDER_ANALYSIS_NAME
        checkpoint = torch.load(run / 'best.pt', map_location='cpu', weights_only=False)
        config = checkpoint['config']
        mode = config.get('preprocessing', 'raw')
        model = ConvAutoencoder(config['latent_dim']).to(device)
        model.load_state_dict(checkpoint['model'])
        model.eval()
        forest = joblib.load(analysis / 'isolation_forest.joblib')
        threshold = json.loads((analysis / 'calibration.json').read_text())['threshold']
        scores = {}
        for variant in transformed[0]:
            latent = []
            with torch.no_grad():
                for offset in range(0, len(sample), 16):
                    x = np.stack([preprocess_array(v[variant], mode) for v in transformed[offset:offset + 16]])[:, None]
                    latent.append(model.encode(torch.from_numpy(x).to(device)).cpu().numpy())
            scores[variant] = -forest.score_samples(np.concatenate(latent))
        pd.DataFrame({'filename': sample.filename, **scores}).to_csv(out / f'{name}_paired_scores.csv', index=False)
        for variant in ['border_strip', 'bright_square', 'dark_square', 'checker']:
            base = scores['jpeg_control' if variant == 'border_strip' else 'original']
            changed = scores[variant]
            delta = changed - base
            rows.append(dict(run=name, preprocessing=mode, test=variant, n=len(sample), median_signed_change=float(np.median(delta)), median_absolute_change=float(np.median(np.abs(delta))), p95_absolute_change=float(np.quantile(np.abs(delta), 0.95)), fraction_score_increased=float((delta > 0).mean()), threshold_flips=int(((base > threshold) != (changed > threshold)).sum()), newly_above_threshold=int(((base <= threshold) & (changed > threshold)).sum()), original_above_threshold=int((base > threshold).sum()), changed_above_threshold=int((changed > threshold).sum())))
        del model, forest, checkpoint
        if device == 'cuda':
            torch.cuda.empty_cache()
        print(name, 'response diagnostics complete', flush=True)
    response = pd.DataFrame(rows)
    response.to_csv(out / 'response_summary.csv', index=False)
    audit = []
    for row in manifest.itertuples():
        with Image.open(row.path) as im:
            a = np.array(im, dtype=np.float32) / 255
        mask = border_near_black_mask(a)
        audit.append({'filename': row.filename, 'source_image_id': row.source_image_id, 'masked_fraction': float(mask.mean()), 'central_defect_mask_fraction': float(mask[101:125, 101:125].mean())})
    audit = pd.DataFrame(audit)
    audit.to_csv(out / 'mask_audit.csv', index=False)
    summary = {'crops': len(audit), 'any_mask': int((audit.masked_fraction > 0).sum()), 'all_masked': int((audit.masked_fraction == 1).sum()), 'over_half_masked': int((audit.masked_fraction > 0.5).sum()), 'median_masked_fraction': float(audit.masked_fraction.median()), 'p95_masked_fraction': float(audit.masked_fraction.quantile(0.95)), 'max_masked_fraction': float(audit.masked_fraction.max())}
    (out / 'mask_summary.json').write_text(json.dumps(summary, indent=2))
    d = {n: json.loads((ROOT / 'outputs' / n / BORDER_ANALYSIS_NAME / 'diagnostics.json').read_text()) for n in ['v3', 'v5', 'v6']}
    jaccard = {n: min((r['flag_jaccard'] or 0 for r in diag['forest_seed_stability'])) for n, diag in d.items()}
    lookup = response.set_index(['run', 'test'])
    border6 = lookup.loc[('v6', 'border_strip'), 'p95_absolute_change']
    criteria = {'border_response_reduced_25_percent_vs_both': bool(all((border6 <= 0.75 * lookup.loc[(n, 'border_strip'), 'p95_absolute_change'] for n in ['v3', 'v5']))), 'all_defect_responses_at_least_70_percent_and_noninferior': bool(all((lookup.loc[('v6', t), 'fraction_score_increased'] >= max(0.7, lookup.loc[('v3', t), 'fraction_score_increased'] - 0.05) for t in ['bright_square', 'dark_square', 'checker']))), 'forest_jaccard_at_least_v3': bool(jaccard['v6'] >= jaccard['v3'])}
    decision = {'criteria': criteria, 'forest_min_jaccard': jaccard, 'numerical_criteria_pass': all(criteria.values()), 'visual_review': 'pending', 'promoted': False, 'note': 'The visual/mask review is still required. Proxy response diagnostics do not establish hidden-anomaly accuracy.'}
    (out / 'decision.json').write_text(json.dumps(decision, indent=2))
    print(response.to_string(index=False))
    print(json.dumps(summary))
    print(json.dumps(decision))

In [ ]:
RERUN_BORDER_CHECKS = False
border_folder = ROOT/'outputs/border_experiment'
if RERUN_BORDER_CHECKS or not (border_folder/'decision.json').exists():
    border_response_checks()
display(pd.read_csv(border_folder/'response_summary.csv'))
display(json.loads((border_folder/'mask_summary.json').read_text()))
display(json.loads((border_folder/'decision.json').read_text()))
print('Synthetic perturbations are diagnostics only; no hidden-label accuracy is inferred.')


## Phase 3.1 — Inspecting the selected crops
I apply the threshold first and then select at most five eligible crops. I never lower it to force five.

For raw-input runs, error compares the original crop with its reconstruction. Standardized panels show the raw crop separately from the model input: their error compares standardized input with reconstruction. The overlay shows location, and is not direct attribution for Isolation Forest.


In [ ]:
selected = pd.read_csv(ANALYSIS/'selected_for_interpretation.csv')
assert len(selected) <= 5 and (selected.novelty_score > calibration['threshold']).all()
display(selected)
for filename in selected.filename:
    display(DisplayImage(filename=str(ANALYSIS/f'heatmap_{Path(filename).stem}.png')))
if selected.empty:
    print('No crop passes the boundary. I do not manufacture a top-five list.')


### Do these exact candidates repeat?
I check the selected filenames against each run's own calibrated flags. This makes disagreement visible at the candidate level. A missing optional run is shown as unavailable rather than treated as a negative result. Agreement still does not establish a genuine anomaly.


In [ ]:
candidate_checks = selected[['filename','source_image_id']].copy()
for name in [spec['name'] for spec in MAIN_RUNS + ROBUSTNESS_RUNS]:
    score_file = ROOT/'outputs'/name/analysis_name/'novelty_scores.csv'
    if score_file.exists():
        other = pd.read_csv(score_file)
        eligible = set(other.loc[other.flagged,'filename'])
        candidate_checks[name] = candidate_checks.filename.isin(eligible)
    else:
        candidate_checks[name] = 'Unavailable'
display(candidate_checks)


## Phase 3.2 — My physical interpretations
Each interpretation follows the same structure: (1) what the heatmap shows spatially, (2) the most plausible physical cause linked to a Genesis Outlier type, (3) an ordinary Martian alternative, and (4) the key limitation. These are hypotheses, not confirmed labels. A new run that selects different crops needs a new visual review.

**Genesis Outlier types from the competition rules:**
- **Type A — Semantic terrain splice:** two terrain patches from different Mars regions stitched together. Expected heatmap: sharp boundary or oblique edge error.
- **Type B — Domain-shifted terrestrial content:** non-Martian image inserted. Expected heatmap: diffuse error across the whole frame.
- **Type C — Synthetic sensor artifact:** hardware-injection simulation. Expected heatmap: stripe, readout-line, or border-region error.


In [ ]:
HYPOTHESES = {'genesis_outlier_types_from_rules': {'note': 'The competition rules state three '
                                              'distinct injected classes: (A) semantic '
                                              'terrain splices, (B) domain-shifted '
                                              'terrestrial content, and (C) synthetic '
                                              'sensor-hardware glitches. These types '
                                              'produce different heatmap signatures. All '
                                              'interpretations below are hypotheses, not '
                                              'confirmed labels.',
                                      'type_A_signature': 'Sharp boundary or edge error '
                                                          'concentrated along a linear '
                                                          'or diagonal divide.',
                                      'type_B_signature': 'Diffuse, spatially '
                                                          'distributed error across the '
                                                          'whole frame â€” the decoder '
                                                          'cannot reconstruct unfamiliar '
                                                          'texture.',
                                      'type_C_signature': 'Localized stripe, '
                                                          'border-region, or '
                                                          'high-frequency artifact error '
                                                          'consistent with a hardware '
                                                          'injection.'},
 'sample_07899.jpg': {'heatmap_pattern': 'Error is distributed diffusely across fine '
                                         'dark mottling on a very bright background. The '
                                         'reconstruction preserves the broad brightness '
                                         'but loses small-scale texture. There is no '
                                         'sharp edge or boundary in the error map.',
                      'primary_hypothesis': 'The decoder has not learned to reconstruct '
                                            'this combination of extreme brightness and '
                                            'fine-grained mottling, producing '
                                            'distributed reconstruction error. This '
                                            'pattern is consistent with either (A) a '
                                            'terrain splice joining a polar-bright '
                                            'region to a different texture, or (B) '
                                            'non-Martian bright content with fine '
                                            'texture that was not seen during training.',
                      'martian_alternative': 'Genuine Martian terrain: a seasonal bright '
                                             'frost mantle over textured polar soil is a '
                                             'plausible ordinary explanation. The '
                                             'latitude metadata is consistent with a '
                                             'polar location, though the coarse supplied '
                                             'value cannot confirm this. Dust mantles at '
                                             'high solar-angle acquisitions also produce '
                                             'high brightness.',
                      'limitation': 'A grayscale crop cannot distinguish ice from dust '
                                    'or from injected bright content. The seasonal and '
                                    'latitudinal interpretation is a hypothesis, not a '
                                    'catalog identification. No spectral data is '
                                    'available.',
                      'reference': 'https://www.uahirise.org/PSP_007712_2670'},
 'sample_08288.jpg': {'heatmap_pattern': 'A pale smooth upper region meets a darker '
                                         'rougher diagonal band in the lower half. The '
                                         'reconstruction smooths the band and larger '
                                         'errors follow the rougher lower region. The '
                                         'boundary between regions is oblique, not '
                                         'aligned with image axes.',
                      'primary_hypothesis': 'The oblique boundary with differently '
                                            'textured regions on each side is consistent '
                                            'with (A) a semantic terrain splice â€” two '
                                            'terrain patches joined at a diagonal seam. '
                                            'The decoder reconstructs both sides '
                                            'imperfectly at the join, producing elevated '
                                            'error there. A natural terrain contact is '
                                            'also possible: a wind-deposited bright '
                                            'surface meeting exposed rougher substrate.',
                      'martian_alternative': 'Wind redistribution of bright dust over '
                                             'rougher terrain is a plausible ordinary '
                                             'Martian process. Relief-dependent '
                                             'illumination across a slope could also '
                                             'produce this contrast without any unusual '
                                             'content.',
                      'limitation': 'The crop provides no reliable slope direction or '
                                    'physical scale. The high latent novelty does not '
                                    'establish that this is an injected sample.',
                      'reference': 'https://www.jpl.nasa.gov/images/pia03789-cerberus-wind-streaks/'},
 'sample_04609.jpg': {'heatmap_pattern': 'Several thin, nearly parallel dark streaks '
                                         'cross a bright smooth background. The decoder '
                                         'blurs the narrow lines and the error map '
                                         'traces those streaks, even though average '
                                         'reconstruction error is low. The error is '
                                         'stripe-shaped and locally concentrated.',
                      'primary_hypothesis': 'Narrow parallel stripes that the decoder '
                                            'cannot reproduce could be consistent with '
                                            '(C) a synthetic sensor-hardware artifact '
                                            'â€” parallel readout lines or injection '
                                            'stripes. Alternatively, this matches (A) a '
                                            'terrain splice where directional dark '
                                            'streaks from a different image were '
                                            'composited onto a bright background.',
                      'martian_alternative': 'Wind-related removal of a bright dust '
                                             'coating exposing darker substrate can '
                                             'produce directional dark streaks on Mars. '
                                             'This is a common ordinary process; '
                                             'orientation alone cannot distinguish a '
                                             'natural wind streak from a hardware stripe '
                                             'or a splice boundary.',
                      'limitation': 'Wind direction and slope cannot be recovered from '
                                    'this crop. Neither hypothesis can be confirmed '
                                    'without reference to the original HiRISE image '
                                    'context.',
                      'reference': 'https://www.jpl.nasa.gov/images/pia07132-wind-streaks-among-flows/'},
 'sample_00840.jpg': {'heatmap_pattern': 'Broad diagonal textured band and scattered '
                                         'small rounded marks on a bright background. '
                                         'Reconstruction preserves the diffuse band but '
                                         'removes small details. Error follows the '
                                         'textured band and local marks.',
                      'primary_hypothesis': 'Diffuse error concentrated on a textured '
                                            'band with small marks is consistent with '
                                            '(B) domain-shifted terrestrial content â€” '
                                            'a non-Martian image whose surface '
                                            'statistics the decoder has not learned. It '
                                            'is also consistent with unusual bright '
                                            'polar terrain the training distribution did '
                                            'not represent well.',
                      'martian_alternative': 'Uneven bright dust cover over a rougher '
                                             'surface could produce this combination of '
                                             'smooth and textured patches. Small '
                                             'depressions could be ordinary surface '
                                             'features without any injected content.',
                      'limitation': 'This crop shares source SRC_154 with two other '
                                    'selected examples. Their shared acquisition '
                                    'conditions may be driving the repeated selection '
                                    'rather than independent anomalous content. No '
                                    'injected object is confirmed by this panel.',
                      'reference': 'https://www.jpl.nasa.gov/images/pia03789-cerberus-wind-streaks/'},
 'sample_05391.jpg': {'heatmap_pattern': 'A rough striated upper region borders a '
                                         'brighter smoother lower area along an oblique '
                                         'boundary. Fine linear textures dominate the '
                                         'upper part; the reconstruction reduces them to '
                                         'broad smooth forms. The error map mainly '
                                         'follows the lost texture in the upper region.',
                      'primary_hypothesis': 'An oblique boundary between two texturally '
                                            'distinct regions, with the decoder failing '
                                            'on the rougher side, is consistent with (A) '
                                            'a semantic splice. Two terrain patches from '
                                            'different source images joined at a '
                                            'diagonal would produce exactly this pattern '
                                            'â€” elevated reconstruction error on the '
                                            'side whose texture departs most from '
                                            'training examples.',
                      'martian_alternative': 'A transition between a dust-mantled '
                                             'surface and more exposed textured '
                                             'substrate is a plausible Martian '
                                             'explanation. Wind erosion or deposition '
                                             'can create such contacts. Relief-dependent '
                                             'illumination is an alternative explanation '
                                             'for the contrast.',
                      'limitation': 'Three of the five selected crops (including this '
                                    'one) come from SRC_154. The repeated selection from '
                                    'one source raises the possibility that acquisition '
                                    'conditions specific to that observation, rather '
                                    'than injected content, are driving elevated novelty '
                                    'scores. This shared-source concern is reported in '
                                    'the diagnostics.',
                      'reference': 'https://www.jpl.nasa.gov/images/pia03789-cerberus-wind-streaks/'}}
for filename in selected.filename:
    item = HYPOTHESES.get(filename)
    if item is None:
        display(Markdown('### '+filename+chr(10)+'New selection: visual review needed.'))
        continue
    parts = [
        '### '+filename,
        '**Heatmap pattern:** '+item.get('heatmap_pattern', item.get('evidence','')),
        '**Primary hypothesis:** '+item.get('primary_hypothesis', item.get('hypothesis','')),
        '**Martian alternative:** '+item.get('martian_alternative', item.get('alternative','')),
        '**Limitation:** '+item.get('limitation',''),
    ]
    if item.get('reference'):
        parts.append('[Background reference — not a crop identification]('+item['reference']+')')
    display(Markdown((chr(10)*2).join(p for p in parts if p.split(':',1)[-1].strip())))


## Phase 4 — What happened in my experiments

### v1: Baseline

I trained a 128-dimensional MSE autoencoder from scratch. Validation MSE was 0.002694; fine texture remained blurred.

### v2 and v3: Loss and capacity

Adding SSIM raised validation SSIM from 0.6611 to 0.6717 with a small MSE cost. Increasing the vector to 256 gave MSE 0.002599 and SSIM 0.6738. More parameters improved reconstruction modestly, not verified anomaly accuracy.

### Threshold and repetition

The initial skew-adjusted fence exceeded every score. I retained it, inspected the multimodal distribution and adopted a fixed mixture envelope. Increasing the forest to 2,000 trees improved repeatability. Independent v3 runs still had flag Jaccards 0.150 and 0.294.

### v4 and v5: Contrast and footprint

Normalization reduced a fixed illumination response, but selected borders became prominent. Excluding connected exact zeros left near-black fragments and weakened forest agreement. I did not promote these variants.

### v6: Near-black border fill

I tested a fixed near-black mask and nearest-valid-pixel fill. Border response decreased, but defect response and forest stability failed the predefined criteria. Filled regions introduced stretched stripes. This was a completed negative experiment, not an abandoned variance mask.

### v7: Gradient loss

I added 0.1 times adjacent-pixel gradient error to 0.9 MSE + 0.1 (1-SSIM). MSE improved 1.68% and gradient error 0.52%. Four of five selected crops were still extremely bright; the darker crop also contains borders. The brightness confound was not solved. The protocol's 0.8 MSE description differed from the actual 0.9 implementation; the saved configuration and source define the executed experiment.

### v8: Geometric augmentation and optional fusion

I added batch flips and quarter-turn rotations to the v7 objective. Validation reconstruction worsened and image-only screening produced zero flags. Lower effective rank does not prove a better representation. A separate post-encoding metadata experiment is reported with its own threshold; zero flags are retained without manufacturing five examples.

### Final decision

I keep v3 as the reference, disclose its brightness and independent-run limitations, and retain all fourteen completed runs as evidence. No hidden-label accuracy is claimed.

## What I would test next
I would test more independent source partitions and longer learning curves under a fixed budget. I would also check whether normalization hides real albedo anomalies. Appearance alone cannot decide that; organizer-held labels would be needed for precision and recall.

## Sources
- [SSIM author resource](https://www.cns.nyu.edu/~lcv/ssim/)
- [Resize-convolution](https://distill.pub/2016/deconv-checkerboard/)
- [Isolation Forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html)
- [Gaussian mixture selection](https://scikit-learn.org/stable/auto_examples/mixture/plot_gmm_selection.html)
- [Connected-region labelling](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.label.html)

The constraints come from the three supplied competition documents. My decision log preserves failed approaches and verification. Team details and private repository access remain submission logistics.
